# Analysis
Analyze a dataset in terms of
- clique distribution per split
- YouTube metadata (optional)


## DVI2
The cleaned version of *Discogs-VI-YT*. This dataset includes a few cliques and versions less than the first iteration of the dataset.

In [61]:
import os

# DVI2 (cleaned Discogs-VI-YT)
dvi2_dir = os.path.join("data", "dvi2", "dataset")
dvi2_path = os.path.join(dvi2_dir, "dvi_cleaned.jsonl")


In [62]:
import json

def read_json_lines(path: str):
    with open(path, "r") as f:
        data = [json.loads(line) for line in f]
    return data

dvi2_full = read_json_lines(dvi2_path)


In [63]:
def read_json(path: str):
    with open(path, "r") as f:
        data = json.load(f)
    return data

splits = ["train", "val", "test"]
data = {}
for split in splits:
    data[split] = read_json(os.path.join(dvi2_dir, f"{split}.json"))
    

In [64]:
import numpy as np

def clique_version_statistics(data):
    """
    Calculate statistics for the number of versions per clique.
    
    Args:
        data (dict): Dictionary containing cliques and their versions.
        
    Returns:
        dict: Statistics including min, max, median, mean, and std of versions per clique.
    """
    total = 0
    n_per_c = []
    print("===================================")
    print(f"{split} set:")
    for clique_id, versions in data[split].items():
        
        n = len(versions)
        total += n
        n_per_c.append(n)

    n_per_c = np.array(n_per_c)

    print(f"{'Cliques:':>20} {len(n_per_c):>10,}")
    print(f"{'Versions:':>20} {total:>10,}")
    print()
    print("Versions per clique:")
    print(f"{'min:':>20} {n_per_c.min():>10}")
    print(f"{'max:':>20} {n_per_c.max():>10}")
    print(f"{'median:':>20} {np.median(n_per_c):>10}")
    print(f"{'mean:':>20} {n_per_c.mean():>10.2f}")
    print(f"{'std:':>20} {n_per_c.std():>10.2f}")
    
for split in splits:
    clique_version_statistics(data)
        

train set:
            Cliques:     74,574
           Versions:    311,804

Versions per clique:
                min:          2
                max:        455
             median:        2.0
               mean:       4.18
                std:       8.40
val set:
            Cliques:      8,294
           Versions:     34,098

Versions per clique:
                min:          2
                max:        258
             median:        2.0
               mean:       4.11
                std:       8.01
test set:
            Cliques:     10,273
           Versions:    111,304

Versions per clique:
                min:          2
                max:        629
             median:        3.0
               mean:      10.83
                std:      27.85


## DVI2-FM
The cleaned and extended dataset. This dataset contains a few less cliques, but drastically more versions

In [65]:

# DVI2-FM (cleaned & extended Discogs-VI-YT)
dvi2fm_dir = os.path.join(dvi2_dir, "matched")
dvi2fm_path = os.path.join(dvi2fm_dir, "dvi_fm_filtered.jsonl")


In [66]:
splits = ["train", "val", "test"]
data = {}
for split in splits:
    data[split] = read_json(os.path.join(dvi2fm_dir, f"{split}.json"))

for split in splits:
    clique_version_statistics(data)
  

train set:
            Cliques:     76,740
           Versions:  1,011,398

Versions per clique:
                min:          2
                max:        483
             median:        5.0
               mean:      13.18
                std:      19.03
val set:
            Cliques:      8,529
           Versions:    110,490

Versions per clique:
                min:          2
                max:        258
             median:        5.0
               mean:      12.95
                std:      18.65
test set:
            Cliques:     10,519
           Versions:    221,480

Versions per clique:
                min:          2
                max:        629
             median:        7.0
               mean:      21.06
                std:      33.71


## YouTube metadata

In [67]:
import pandas as pd

def read_dataset_split(data: dict):
    rows = []
    for clique_id, versions in data.items():
        for item in versions:
            item_with_clique = {"clique_id": clique_id, **item}
            rows.append(item_with_clique)
    return pd.DataFrame(rows)

df = pd.DataFrame()
for split in splits:
    df_tmp = read_dataset_split(data[split])
    df_tmp["split"] = split
    df = pd.concat([df, df_tmp], ignore_index=True)


In [68]:
import pandas as pd

metadata = pd.read_json("data/metadata_filtered.jsonl", lines=True, orient='records')
metadata = metadata.loc[metadata.id.isin(df.youtube_id),:]

new_columns = ["clique_id"] + metadata.columns.tolist()
metadata = pd.merge(
    df,
    metadata,
    left_on="youtube_id",
    right_on="id",
    how="left",
)
metadata = metadata[new_columns]


In [69]:
metadata

,clique_id,type,id,title,publishedTime,duration,viewCount,thumbnails,richThumbnail,descriptionSnippet,channel,accessibility,link,shelfTitle
0,C-0000000_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,C-0000000_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,C-0000006_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,C-0000006_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,C-0000006_0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1343363,C-0348564_0,video,rLQ7PvzI6PM,Sweeter Than Fiction - Taylor Swift (Mini Cove...,11 years ago,2:34,"{'text': '15,435 views', 'short': '15K views'}",[{'url': 'https://i.ytimg.com/vi/rLQ7PvzI6PM/h...,{'url': 'https://i.ytimg.com/an_webp/rLQ7PvzI6...,[{'text': 'so you guys know there's NO WAY i c...,"{'name': 'for3v3rfaithful', 'id': 'UCFifQL1z9q...",{'title': 'Sweeter Than Fiction - Taylor Swift...,https://www.youtube.com/watch?v=rLQ7PvzI6PM,None
1343364,C-0348564_0,video,uNMfbJJUgOI,Taylor Swift Sweeter than Fiction Vocals {Lyri...,2 years ago,0:18,"{'text': '6,243 views', 'short': '6.2K views'}",[{'url': 'https://i.ytimg.com/vi/uNMfbJJUgOI/h...,{'url': 'https://i.ytimg.com/an_webp/uNMfbJJUg...,None,"{'name': 'Celeb Stuff', 'id': 'UClsFDfEk8CQWOz...",{'title': 'Taylor Swift Sweeter than Fiction V...,https://www.youtube.com/watch?v=uNMfbJJUgOI,None
1343365,C-0348564_0,video,yfnIBgpIh-k,Taylor Swift -Sweeter Than Fiction (soundtrack...,11 years ago,3:59,"{'text': '397 views', 'short': '397 views'}",[{'url': 'https://i.ytimg.com/vi/yfnIBgpIh-k/h...,None,[{'text': 'I do not own any of this.'}],"{'name': 'Linh Phoung', 'id': 'UCD32jt8FNxlIwb...",{'title': 'Taylor Swift -Sweeter Than Fiction ...,https://www.youtube.com/watch?v=yfnIBgpIh-k,None
1343366,C-0348564_0,video,ytGqUs3UIPk,"Sweeter Than Fiction (From ""One Chance"" Soundt...",None,3:58,"{'text': '2,261,546 views', 'short': '2.2M vie...",[{'url': 'https://i.ytimg.com/vi/ytGqUs3UIPk/h...,None,[{'text': 'Provided to YouTube by Universal Mu...,"{'name': 'Taylor Swift', 'id': 'UCqECaJ8Gagnn7...","{'title': 'Sweeter Than Fiction (From ""One Cha...",https://www.youtube.com/watch?v=ytGqUs3UIPk,None
